In [1]:
from helpers.pipeline import load_qa_pairs, load_snippets

df_questions = load_qa_pairs()
df_questions.head()

,id,question,ground truth,expected answer,srcs,annotator_1_notes,annotator_1_label,annotator_2_notes,annotator_2_label,annotator_3_notes,annotator_3_label
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),"['But in 1974, a military junta known as the D...","This paragraph contains minor issue: the ""peac...",Minor Issue(s),This is a correct account of Ethiopia's histro...,No Issues,The claims made in the Paragraph about the Eth...,No Issues
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,['Movies have long been a powerful force in sh...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,['Movies have long been a powerful force in sh...,"This paragraph is mostly accurate, but there's...",Minor Issue(s),The claim regarding 12% of female characters w...,Minor Issue(s),The claims are generally true. But I could not...,Not Sure
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,['Indira Gandhi was the first and only woman t...,There are minor inaccuracies in this paragraph...,Minor Issue(s),Indira became member of the Parliament in 1964...,Minor Issue(s),Most of the biography about Indira Gandhi is a...,Minor Issue(s)
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,['Grace: Can you tell me a little bit about it...,Tea was actually introduced in North America i...,Minor Issue(s),Tea was actually introduced in North America i...,Minor Issue(s),The history of tea in China is accurate. Howev...,Minor Issue(s)


In [2]:
# Dictionary mapping models and methods to their specific snippet URLs
snippet_urls = {
    "llama3:8b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/107/raw/main/llama3_8b_baseline_answers.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/108/raw/main/llama3_8b_icl_answers.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/84/raw/main/answers_sft_llama.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/87/raw/main/answers_peft_llama.csv"
    },
    "gemma3:4b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/105/raw/main/gemma3_4b_baseline_answers.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/106/raw/main/gemma3_4b_icl_answers.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/83/raw/main/answers_sft_gemma.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/86/raw/main/answers_peft_gemma.csv"
    },
    "deepseek-r1:8b": {
        "baseline" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/103/raw/main/deepseek-r1_8b_baseline_answers.csv",
        "icl" : "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/104/raw/main/deepseek-r1_8b_icl_answers.csv",
        "sft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/85/raw/main/answers_sft_deepseek.csv",
        "peft": "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/88/raw/main/answers_peft_deepseek.csv"
    }
}

dfs = load_snippets(snippet_urls)

# Access a specific one:
dfs['llama3:8b_baseline'].head()

Loaded llama3:8b_baseline: (91, 6)
Loaded llama3:8b_icl: (91, 6)
Loaded llama3:8b_sft: (91, 6)
Loaded llama3:8b_peft: (91, 6)
Loaded gemma3:4b_baseline: (91, 6)
Loaded gemma3:4b_icl: (91, 6)
Loaded gemma3:4b_sft: (91, 6)
Loaded gemma3:4b_peft: (91, 6)
Loaded deepseek-r1:8b_baseline: (91, 6)
Loaded deepseek-r1:8b_icl: (91, 6)
Loaded deepseek-r1:8b_sft: (91, 6)
Loaded deepseek-r1:8b_peft: (91, 6)


,id,question,ground truth,incorrect answer,model answer,reasoning
0,ethiopia_challenges__btithihhtt,Who did Prime Minister Abiy Ahmed of Ethiopeia...,Eritrea,Tigray People's Liberation Front (TPLF),Prime Minister Abiy Ahmed of Ethiopia had peac...,NaN
1,topic_260__mtaftfttit,How many female characters had speaking roles ...,36.3%,30%,According to the Geena Davis Institute study f...,NaN
2,topic_260__mtaftfttit,Does the Geena Davis Institute study from 2017...,"No, the Geena Davis Institute does not explici...",12%,"Yes, the Geena Davis Institute's study from 20...",NaN
3,topic_493__isitgsgshgsg,When did Indira Gandhi become a member of the ...,1964,1955,Indira Gandhi became a member of the Indian Pa...,NaN
4,topic_131__gtttgtiigt,When was tea introduced to North America?,17th century,19th century,Tea was introduced to North America in the ear...,NaN


In [ ]:
import pandas as pd
from helpers.pipeline import assess_response_quality
from tqdm.notebook import tqdm

pbar = tqdm(dfs.items())
for model_method, df in pbar:
    pbar.set_description(f"Evaluating: {model_method}")
    model, method = model_method.split("_")
    scores = assess_response_quality("all-models", df, verbose=True)
    scores = pd.Series(scores, name="score")
    evaluation = pd.concat([df, scores], axis=1)
    evaluation.to_csv(f"../data/{model_method.replace(":", "_")}_evaluation.csv", index=False)
